<!--BOOK_INFORMATION-->
<img align="left" style="padding-right:10px;" src="figures/PDSH-cover-small.png">
*This notebook contains an excerpt from the [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) by Jake VanderPlas; the content is available [on GitHub](https://github.com/jakevdp/PythonDataScienceHandbook).*

*The text is released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT). If you find this content useful, please consider supporting the work by [buying the book](http://shop.oreilly.com/product/0636920034919.do)!*

<!--NAVIGATION-->
< [Handling Missing Data](03.04-Missing-Values.ipynb) | [Contents](Index.ipynb) | [Combining Datasets: Concat and Append](03.06-Concat-And-Append.ipynb) >

# 3.6 Hierarchical Indexing / 层级索引

Up to this point we've been focused primarily on one-dimensional and two-dimensional data, stored in Pandas ``Series`` and ``DataFrame`` objects, respectively.
Often it is useful to go beyond this and store higher-dimensional data–that is, data indexed by more than one or two keys.
While Pandas does provide ``Panel`` and ``Panel4D`` objects that natively handle three-dimensional and four-dimensional data (see [Aside: Panel Data](#Aside:-Panel-Data)), a far more common pattern in practice is to make use of *hierarchical indexing* (also known as *multi-indexing*) to incorporate multiple index *levels* within a single index.
In this way, higher-dimensional data can be compactly represented within the familiar one-dimensional ``Series`` and two-dimensional ``DataFrame`` objects.

🐍 实践中，通过层级索引（也称多级索引）配合多个不同登记一级索引一起使用，可以将高维数组转换成类似一维Series和二维DataFrame对象的形式

In this section, we'll explore the direct creation of ``MultiIndex`` objects, considerations when indexing, slicing, and computing statistics across multiply indexed data, and useful routines for converting between simple and hierarchically indexed representations of your data.

We begin with the standard imports:

In [1]:
import pandas as pd
import numpy as np

## 3.6.1 A Multiply Indexed Series / 多级索引Series

Let's start by considering how we might represent two-dimensional data within a one-dimensional ``Series``.
For concreteness, we will consider a series of data where each point has a character and numerical key.。。

### 1. The bad way / 笨办法

Suppose you would like to track data about states from two different years.
Using the Pandas tools we've already covered, you might be tempted to simply use Python tuples as keys:

In [2]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)
pop

(California, 2000)    33871648
(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
(Texas, 2010)         25145561
dtype: int64

With this indexing scheme, you can straightforwardly index or slice the series based on this multiple index:

In [3]:
# 🐍 在Series数组中，使用切片操作

pop[('California', 2010):('Texas', 2000)]

(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
dtype: int64

But the convenience ends there. For example, if you need to select all values from 2010, you'll need to do some messy (and potentially slow) munging to make it happen:

In [4]:
# 🐍 for循环，行索引中使用if条件判断，元组中 [1]位 的值为2010

pop[[i for i in pop.index if i[1] == 2000]]

(California, 2000)    33871648
(New York, 2000)      18976457
(Texas, 2000)         20851820
dtype: int64

This produces the desired result, but is not as clean (or as efficient for large datasets) as the slicing syntax we've grown to love in Pandas.

### 2. The Better Way: Pandas MultiIndex / 好办法：Pandas多级索引
Fortunately, Pandas provides a better way.
Our tuple-based indexing is essentially a rudimentary multi-index, and the Pandas ``MultiIndex`` type gives us the type of operations we wish to have.
We can create a multi-index from the tuples as follows:

In [5]:
# 🐍 Pandas的MultiIndex类型，使用元组表示索引是多级索引的基础

index = pd.MultiIndex.from_tuples(index)
index

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           )

In [6]:
# 获取MultiIndex对象的帮助文档

help(index)

Help on MultiIndex in module pandas.core.indexes.multi object:

class MultiIndex(pandas.core.indexes.base.Index)
 |  MultiIndex(levels=None, codes=None, sortorder=None, names=None, dtype=None, copy: 'bool' = False, name=None, verify_integrity: 'bool' = True) -> 'MultiIndex'
 |  
 |  A multi-level, or hierarchical, index object for pandas objects.
 |  
 |  Parameters
 |  ----------
 |  levels : sequence of arrays
 |      The unique labels for each level.
 |  codes : sequence of arrays
 |      Integers for each level designating which label at each location.
 |  sortorder : optional int
 |      Level of sortedness (must be lexicographically sorted by that
 |      level).
 |  names : optional sequence of objects
 |      Names for each of the index levels. (name is accepted for compat).
 |  copy : bool, default False
 |      Copy the meta-data.
 |  verify_integrity : bool, default True
 |      Check that the levels/codes are consistent and valid.
 |  
 |  Attributes
 |  ----------
 |  name

Notice that the ``MultiIndex`` contains multiple *levels* of indexing–in this case, the state names and the years, as well as multiple *labels* for each data point which encode these levels.

If we re-index our series with this ``MultiIndex``, we see the hierarchical representation of the data:

In [7]:
# 🐍 索引重置 reindex --> 层级索引(pd.Series)

pop = pop.reindex(index)
pop

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

Here the first two columns of the ``Series`` representation show the multiple index values, while the third column shows the data.
Notice that some entries are missing in the first column: in this multi-index representation, any blank entry indicates the same value as the line above it.

Now to access all data for which the second index is 2010, we can simply use the Pandas slicing notation:

In [8]:
# 🐍 直接使用第二个索引值(2010), 与Pandas切片的法一致

pop[:, 2000]

California    33871648
New York      18976457
Texas         20851820
dtype: int64

The result is a singly indexed array with just the keys we're interested in.
This syntax is much more convenient (and the operation is much more efficient!) than the home-spun tuple-based multi-indexing solution that we started with.
We'll now further discuss this sort of indexing operation on hieararchically indexed data.

### 3. MultiIndex as extra dimension / 高维数据的多级索引

You might notice something else here: we could easily have stored the same data using a simple ``DataFrame`` with index and column labels.
In fact, Pandas is built with this equivalence in mind. The ``unstack()`` method will quickly convert a multiply indexed ``Series`` into a conventionally indexed ``DataFrame``:

In [17]:
pop

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [19]:
# 🐍 unstack()方法，可以快速将一个多级索引的Series转化为普通索引的DataFrame
# 🐍 多级(Series) --> 多维(DataFrame)

pop_df1 = pop.unstack()
pop_df1

,2000,2010
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


In [12]:
# reindex()方法，可以添加index

pop_df.reindex(index)

2000  2010
California 2000   NaN   NaN
           2010   NaN   NaN
New York   2000   NaN   NaN
           2010   NaN   NaN
Texas      2000   NaN   NaN
           2010   NaN   NaN

Naturally, the ``stack()`` method provides the opposite operation:

In [20]:
# 🐍 stack()方法，与unstack()方法效果相反
# 🐍 多维(DataFrame) --> 多级(Series)

pop_df1.stack()

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

Seeing this, you might wonder why would we would bother with hierarchical indexing at all.
The reason is simple: just as we were able to use multi-indexing to represent two-dimensional data within a one-dimensional ``Series``, we can also use it to represent data of three or more dimensions in a ``Series`` or ``DataFrame``.
Each extra level in a multi-index represents an extra dimension of data; taking advantage of this property gives us much more flexibility in the types of data we can represent. Concretely, we might want to add another column of demographic data for each state at each year (say, population under 18) ; with a ``MultiIndex`` this is as easy as adding another column to the ``DataFrame``:

🐍 多级索引每增加一级，就表示数据增加一维，利用这一特点就可以轻松表示任意维度的数据了

In [21]:
# 🐍 对MultiIndex的对象，类似DataFrame的操作，增加一列 under18

pop_df2 = pd.DataFrame({'total': pop,
                       'under18': [9267089, 9284094,
                                   4687374, 4318033,
                                   5906301, 6879014]})
pop_df2

total  under18
California 2000  33871648  9267089
           2010  37253956  9284094
New York   2000  18976457  4687374
           2010  19378102  4318033
Texas      2000  20851820  5906301
           2010  25145561  6879014

In [22]:
# 多维索引的 DataFrame，压缩成多级索引的 Series

pop_ser = pop_df2.stack()
pop_ser

California  2000  total      33871648
                  under18     9267089
            2010  total      37253956
                  under18     9284094
New York    2000  total      18976457
                  under18     4687374
            2010  total      19378102
                  under18     4318033
Texas       2000  total      20851820
                  under18     5906301
            2010  total      25145561
                  under18     6879014
dtype: int64

In [81]:
# 参考Jupyter中控台的输出结果

pop_ser.shape
# type(pop_ser)
# pop_df.shape

(12,)

In addition, all the ufuncs and other functionality discussed in [Operating on Data in Pandas](03.03-Operations-in-Pandas.ipynb) work with hierarchical indices as well.
Here we compute the fraction of people under 18 by year, given the above data:

In [23]:
f_u18 = pop_df2['under18'] / pop_df2['total']
f_u18.unstack()

,2000,2010
California,0.273594,0.249211
New York,0.247010,0.222831
Texas,0.283251,0.273568


This allows us to easily and quickly manipulate and explore even high-dimensional data.

## 3.6.2 Methods of MultiIndex Creation / 多级索引的创建方法

The most straightforward way to construct a multiply indexed ``Series`` or ``DataFrame`` is to simply pass a list of two or more index arrays to the constructor. For example:

In [24]:
# 🐍 创建多级索引最直接的办法，将Index参数设置为至少二维的索引数组

df = pd.DataFrame(np.random.rand(4, 2),
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
df

data1     data2
a 1  0.132427  0.488839
  2  0.420281  0.679179
b 1  0.737644  0.750668
  2  0.600881  0.443070

The work of creating the ``MultiIndex`` is done in the background.

Similarly, if you pass a dictionary with appropriate tuples as keys, Pandas will automatically recognize this and use a ``MultiIndex`` by default:

In [25]:
# 🐍 把元组作为键的字典传递给Pandas，也会默认转换为MultiIndex

data = {('California', 2000): 33871648,
        ('California', 2010): 37253956,
        ('Texas', 2000): 20851820,
        ('Texas', 2010): 25145561,
        ('New York', 2000): 18976457,
        ('New York', 2010): 19378102}
pd.Series(data)

California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
New York    2000    18976457
            2010    19378102
dtype: int64

Nevertheless, it is sometimes useful to explicitly create a ``MultiIndex``; we'll see a couple of these methods here.

### 1. Explicit MultiIndex constructors / 显式的创建多级索引

For more flexibility in how the index is constructed, you can instead use the class method constructors available in the ``pd.MultiIndex``.
For example, as we did before, you can construct the ``MultiIndex`` from a simple list of arrays giving the index values within each level:

In [26]:
# 在Python中，多级索引MultiIndex对象在程序中怎么使用?

import pandas as pd

# 创建一个具有多级行索引的示例DataFrame
data = {
    'A': [1, 2, 3, 4],
    'B': [5, 6, 7, 8]
}

index = pd.MultiIndex.from_tuples([('Group1', 'A'), ('Group1', 'B'), ('Group2', 'A'), ('Group2', 'B')], names=['Group', 'Variable'])

index
# df = pd.DataFrame(data, index=index)


MultiIndex([('Group1', 'A'),
            ('Group1', 'B'),
            ('Group2', 'A'),
            ('Group2', 'B')],
           names=['Group', 'Variable'])

In [27]:
# 🐍 使用pd.MultiIndex中的类方法构建多级索引
# 🐍 通过不同等级的若干简单数组组成的列表构建MultiIndex

index1 = pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])
index1

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [28]:
data_df = pd.DataFrame(range(4), index=index1)
data_df

0
a 1  0
  2  1
b 1  2
  2  3

In [29]:
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2],['x', 'y', 'z', 'n']])

MultiIndex([('a', 1, 'x'),
            ('a', 2, 'y'),
            ('b', 1, 'z'),
            ('b', 2, 'n')],
           )

You can construct it from a list of tuples giving the multiple index values of each point:

In [30]:
# 🐍 通过包含多个索引值的元组构成的列表创建MultiIndex

pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [31]:
pd.MultiIndex.from_tuples([('a', 1, 'x'), ('a', 2, 'y'), ('b', 1, 'z'), ('b', 2, 'n')])

MultiIndex([('a', 1, 'x'),
            ('a', 2, 'y'),
            ('b', 1, 'z'),
            ('b', 2, 'n')],
           )

You can even construct it from a Cartesian product of single indices:

In [32]:
# 🐍 用两个索引的笛卡尔积（Cartesian product）创建MultiIndex

index = pd.MultiIndex.from_product([['a', 'b'], [1, 2]])

df_x = pd.DataFrame(np.random.rand(4, 1), index=index, columns=['abc'])

df_x

abc
a 1  0.032389
  2  0.021360
b 1  0.116396
  2  0.063332

In [33]:
pd.MultiIndex.from_product([['a', 'b'], [1, 2], ['x', 'y', 'z', 'n']])

MultiIndex([('a', 1, 'x'),
            ('a', 1, 'y'),
            ('a', 1, 'z'),
            ('a', 1, 'n'),
            ('a', 2, 'x'),
            ('a', 2, 'y'),
            ('a', 2, 'z'),
            ('a', 2, 'n'),
            ('b', 1, 'x'),
            ('b', 1, 'y'),
            ('b', 1, 'z'),
            ('b', 1, 'n'),
            ('b', 2, 'x'),
            ('b', 2, 'y'),
            ('b', 2, 'z'),
            ('b', 2, 'n')],
           )

Similarly, you can construct the ``MultiIndex`` directly using its internal encoding by passing ``levels`` (a list of lists containing available index values for each level) and ``labels`` (a list of lists that reference these labels):

In [34]:
# 🐍 更可以直接提供levels（包含每个等级的索引值列表的列表）和labels（包含每个索引值标签列表的列表）创建MultiIndex
# 🐍 pd.MultiIndex构造函数不接受labels参数，而是接受codes参数来指定每个级别的标签索引值 !!!

pd.MultiIndex(levels=[['a', 'b'], [1, 2]],
              codes=[[0, 0, 1, 1], [0, 1, 0, 1]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [35]:
# 🐍 codes的组合数为：8 = 2^2^2 (levels每级水平数)

pd.MultiIndex(levels=[['a', 'b'], [1, 2], ['x', 'y']],
              codes=[[0, 1, 0, 1, 0, 1, 0, 1], [0, 0, 0, 0, 1, 1, 1, 1], [0, 1, 0, 1, 0, 1, 0, 1]])

MultiIndex([('a', 1, 'x'),
            ('b', 1, 'y'),
            ('a', 1, 'x'),
            ('b', 1, 'y'),
            ('a', 2, 'x'),
            ('b', 2, 'y'),
            ('a', 2, 'x'),
            ('b', 2, 'y')],
           )

In [36]:
# 🐍 codes的组合数为：16 = 2*2*4 (levels每级水平数)

pd.MultiIndex(levels=[['a', 'b'], [1, 2], ['x', 'y', 'z', 'n']],
              codes=[[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], 
                     [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1], 
                     [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3],
                     ]
              )

MultiIndex([('a', 1, 'x'),
            ('a', 1, 'y'),
            ('a', 1, 'z'),
            ('a', 1, 'n'),
            ('a', 2, 'x'),
            ('a', 2, 'y'),
            ('a', 2, 'z'),
            ('a', 2, 'n'),
            ('b', 1, 'x'),
            ('b', 1, 'y'),
            ('b', 1, 'z'),
            ('b', 1, 'n'),
            ('b', 2, 'x'),
            ('b', 2, 'y'),
            ('b', 2, 'z'),
            ('b', 2, 'n')],
           )

In [37]:
# 🐍 codes的组合数为：16 = 2*2*4 (levels每级水平数)

index = pd.MultiIndex.from_product([['a', 'b'], [1, 2], ['x', 'y', 'z', 'n']]),

index

(MultiIndex([('a', 1, 'x'),
             ('a', 1, 'y'),
             ('a', 1, 'z'),
             ('a', 1, 'n'),
             ('a', 2, 'x'),
             ('a', 2, 'y'),
             ('a', 2, 'z'),
             ('a', 2, 'n'),
             ('b', 1, 'x'),
             ('b', 1, 'y'),
             ('b', 1, 'z'),
             ('b', 1, 'n'),
             ('b', 2, 'x'),
             ('b', 2, 'y'),
             ('b', 2, 'z'),
             ('b', 2, 'n')],
            ),)

Any of these objects can be passed as the ``index`` argument when creating a ``Series`` or ``Dataframe``, or be passed to the ``reindex`` method of an existing ``Series`` or ``DataFrame``.

### 2. MultiIndex level names / 多级索引的等级名称

Sometimes it is convenient to name the levels of the ``MultiIndex``.
This can be accomplished by passing the ``names`` argument to any of the above ``MultiIndex`` constructors, or by setting the ``names`` attribute of the index after the fact:

In [45]:
# 🐍 给MultiIndex的等级添加等级名称，通过names参数
# 🐍 运行此代码，需要pop数组为多级索引数组(参考上前面stack命令)

pop.index.names = ['state', 'year']
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

With more involved datasets, this can be a useful way to keep track of the meaning of various index values.

### 3. MultiIndex for columns / 多级列索引

In a ``DataFrame``, the rows and columns are completely symmetric, and just as the rows can have multiple levels of indices, the columns can have multiple levels as well.
Consider the following, which is a mock-up of some (somewhat realistic) medical data:

In [39]:
import numpy as np
import pandas as pd

In [75]:
# 🐍 hierarchical indices and columns / 多级行列索引

index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# mock some data / 创建一个4x6数组，保留小数点后一位， round(xx, 1)
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

# create the DataFrame
health_data = pd.DataFrame(data, index=index, columns=columns)
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      45.0  37.8  18.0  36.1  45.0  36.3
     2      37.0  37.4  42.0  36.5  36.0  37.0
2014 1      22.0  37.0  41.0  36.0  32.0  37.1
     2      19.0  36.9  47.0  36.4  35.0  39.1

Here we see where the multi-indexing for both rows and columns can come in *very* handy.
This is fundamentally four-dimensional data, where the dimensions are the subject, the measurement type, the year, and the visit number.
With this in place we can, for example, index the top-level column by the person's name and get a full ``DataFrame`` containing just that person's information:

In [41]:
# 🐍 在列索引的第一级查询姓名，从而获取包含一个人（例如Guido）全部检查信息的DataFrame

health_data['Guido']

type          HR  Temp
year visit            
2013 1      21.0  37.9
     2      55.0  36.4
2014 1      51.0  36.9
     2      42.0  37.4

For complicated records containing multiple labeled measurements across multiple times for many subjects (people, countries, cities, etc.) use of hierarchical rows and columns can be extremely convenient!

## 3.6.3 Indexing and Slicing a MultiIndex / 多级索引的取值和切片

Indexing and slicing on a ``MultiIndex`` is designed to be intuitive, and it helps if you think about the indices as added dimensions.
We'll first look at indexing multiply indexed ``Series``, and then multiply-indexed ``DataFrame``s.

### 1. Multiply indexed Series / Series多级索引

Consider the multiply indexed ``Series`` of state populations we saw earlier:

In [42]:
# 🐍 多级索引的Series数组
# 🐍 Series的基本索引是行索引

pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

We can access single elements by indexing with multiple terms:

In [43]:
# 🐍 对多级索引值获单个元素

pop['California', 2000]

33871648

The ``MultiIndex`` also supports *partial indexing*, or indexing just one of the levels in the index.
The result is another ``Series``, with the lower-level indices maintained:

In [46]:
# 🐍 MultiIndex支持局部取值，即只取索引的某一个层级
# 🐍 如只取最高级的索引，低层级索引值会被保留，获得一个新的Series

pop['California']

year
2000    33871648
2010    37253956
dtype: int64

In [47]:
pop[::2]

state       year
California  2000    33871648
New York    2000    18976457
Texas       2000    20851820
dtype: int64

Partial slicing is available as well, as long as the ``MultiIndex`` is sorted (see discussion in [Sorted and Unsorted Indices](#Sorted-and-unsorted-indices)):

In [48]:
# 🐍 局部切片，要求MultiIndex按照顺序排列

pop.loc['California':'New York']

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
dtype: int64

With sorted indices, partial indexing can be performed on lower levels by passing an empty slice in the first index:

In [49]:
# 🐍 如果索引已经排序，第一层索引可以使用空切片，直接用第二层级索引取值

pop[:, 2000]

state
California    33871648
New York      18976457
Texas         20851820
dtype: int64

Other types of indexing and selection (discussed in [Data Indexing and Selection](03.02-Data-Indexing-and-Selection.ipynb)) work as well; for example, selection based on Boolean masks:

In [50]:
# 🐍 布尔掩码索引取值

pop[pop > 22000000]

state       year
California  2000    33871648
            2010    37253956
Texas       2010    25145561
dtype: int64

Selection based on fancy indexing also works:

In [51]:
# 🐍 花哨的索引取值

pop[['California', 'Texas']]

state       year
California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
dtype: int64

### 2. Multiply indexed DataFrames / DataFrame多级索引

A multiply indexed ``DataFrame`` behaves in a similar manner.
Consider our toy medical ``DataFrame`` from before:

In [52]:
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      38.0  36.6  21.0  37.9  36.0  37.7
     2      29.0  37.5  55.0  36.4  47.0  37.3
2014 1      23.0  36.2  51.0  36.9  30.0  35.2
     2      44.0  37.4  42.0  37.4  21.0  36.0

Remember that columns are primary in a ``DataFrame``, and the syntax used for multiply indexed ``Series`` applies to the columns.
For example, we can recover Guido's heart rate data with a simple operation:

In [53]:
# 🐍 DataFrame的基本索引是列索引，因此Series中的多级索引用法在DataFrame中就应用到列上了

health_data['Guido', 'HR']

year  visit
2013  1        21.0
      2        55.0
2014  1        51.0
      2        42.0
Name: (Guido, HR), dtype: float64

Also, as with the single-index case, we can use the ``loc``, ``iloc``, and ``ix`` indexers introduced in [Data Indexing and Selection](03.02-Data-Indexing-and-Selection.ipynb). For example:

In [54]:
# 🐍 与单索引类似，DataFrame的多级索引同样可以使用索引器loc, iloc, 参见§3.3

health_data.iloc[:2, :2]

subject      Bob      
type          HR  Temp
year visit            
2013 1      38.0  36.6
     2      29.0  37.5

These indexers provide an array-like view of the underlying two-dimensional data, but each individual index in ``loc`` or ``iloc`` can be passed a tuple of multiple indices. For example:

In [55]:
# 🐍 索引器loc, iloc可以传递多个层级的索引元组

health_data.loc[:, ('Bob', 'HR')]

year  visit
2013  1        38.0
      2        29.0
2014  1        23.0
      2        44.0
Name: (Bob, HR), dtype: float64

Working with slices within these index tuples is not especially convenient; trying to create a slice within a tuple will lead to a syntax error:

In [56]:
# 🐍 索引元组的用法中，如果在元组中使用切片-->导致错误

health_data.loc[(:, 1), (:, 'HR')]

SyntaxError: invalid syntax (3424465027.py, line 3)

You could get around this by building the desired slice explicitly using Python's built-in ``slice()`` function, but a better way in this context is to use an ``IndexSlice`` object, which Pandas provides for precisely this situation.
For example:

In [57]:
# 🐍 Python内置的slice()函数可以获取切片
# 🐍 Pandas专门使用 IndexSlice对象，解决元组切片问题

idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[:, 'HR']]

,subject,Bob,Guido,Sue
,type,HR,HR,HR
year,visit,,,
2013,1,38.0,21.0,36.0
2014,1,23.0,51.0,30.0


There are so many ways to interact with data in multiply indexed ``Series`` and ``DataFrame``s, and as with many tools in this book the best way to become familiar with them is to try them out!

## 3.6.4 Rearranging Multi-Indices / 多级索引行列转换

One of the keys to working with multiply indexed data is knowing how to effectively transform the data.
There are a number of operations that will preserve all the information in the dataset, but rearrange it for the purposes of various computations.
We saw a brief example of this in the ``stack()`` and ``unstack()`` methods, but there are many more ways to finely control the rearrangement of data between hierarchical indices and columns, and we'll explore them here.

🐍 使用多级索引的关键是掌握有效的数据转换的方法

### 1. Sorted and unsorted indices / 有序的索引和无序的索引

Earlier, we briefly mentioned a caveat, but we should emphasize it more here.
*Many of the ``MultiIndex`` slicing operations will fail if the index is not sorted.*
Let's take a look at this here.

We'll start by creating some simple multiply indexed data where the indices are *not lexographically sorted*:

🐍 如果MultiIndex不是有序的索引，大多数切片操作都会失败

In [58]:
# 🐍 创建一个无序的多级索引Series数组 a-c-b

index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']
data

char  int
a     1      0.619654
      2      0.392711
c     1      0.745681
      2      0.517556
b     1      0.413288
      2      0.796054
dtype: float64

In [59]:
data['a':'b']

UnsortedIndexError: 'Key length (1) was greater than MultiIndex lexsort depth (0)'

If we try to take a partial slice of this index, it will result in an error:

In [60]:
try:
    data['a':'b']
except KeyError as e:
    print(type(e))
    print(e)

<class 'pandas.errors.UnsortedIndexError'>
'Key length (1) was greater than MultiIndex lexsort depth (0)'


Although it is not entirely clear from the error message, this is the result of the MultiIndex not being sorted.
For various reasons, partial slices and other similar operations require the levels in the ``MultiIndex`` to be in sorted (i.e., lexographical) order.
Pandas provides a number of convenience routines to perform this type of sorting; examples are the ``sort_index()`` and ``sortlevel()`` methods of the ``DataFrame``.
We'll use the simplest, ``sort_index()``, here:

🐍 局部切片和其它相似操作都要求MultiIndex的各级索引是有序的(字典顺序A-Z)

In [61]:
# 🐍 Pandas方法sort_index(), 给MultiIndex对象排序

data = data.sort_index()
data

char  int
a     1      0.619654
      2      0.392711
b     1      0.413288
      2      0.796054
c     1      0.745681
      2      0.517556
dtype: float64

With the index sorted in this way, partial slicing will work as expected:

In [62]:
data['a':'b']

char  int
a     1      0.619654
      2      0.392711
b     1      0.413288
      2      0.796054
dtype: float64

### 2. Stacking and unstacking indices / 索引 stack 与 unstack

As we saw briefly before, it is possible to convert a dataset from a stacked multi-index to a simple two-dimensional representation, optionally specifying the level to use:

In [63]:
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [64]:
# 🐍 多级索引数据集转换，可以通过 level 参数设置索引层级
# 🐍 level 代表的 Index 层级，此例中为State名称列表

level0 = pop.unstack(level=0)
level0

state,California,New York,Texas
year,,,
2000,33871648,18976457,20851820
2010,37253956,19378102,25145561


In [65]:
level1 =  pop.unstack(level=1)
level1

year,2000,2010
state,,
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


The opposite of ``unstack()`` is ``stack()``, which here can be used to recover the original series:

In [66]:
# 🐍 unstack() 是 stack() 的逆操作，同时使用，数据保持不变

pop.unstack().stack()

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

### 3. Index setting and resetting / 索引的设置与重置

Another way to rearrange hierarchical data is to turn the index labels into columns; this can be accomplished with the ``reset_index`` method.
Calling this on the population dictionary will result in a ``DataFrame`` with a *state* and *year* column holding the information that was formerly in the index.
For clarity, we can optionally specify the name of the data for the column representation:

In [67]:
# 🐍 行列标签转换，通过 reset_index 方法实现

pop_flat = pop.reset_index(name='population')
pop_flat # DataFrame shape(6,3)

,state,year,population
0,California,2000,33871648
1,California,2010,37253956
2,New York,2000,18976457
3,New York,2010,19378102
4,Texas,2000,20851820
5,Texas,2010,25145561


Often when working with data in the real world, the raw input data looks like this and it's useful to build a ``MultiIndex`` from the column values.
This can be done with the ``set_index`` method of the ``DataFrame``, which returns a multiply indexed ``DataFrame``:

In [68]:
# 🐍 DataFrame 的 set_index() 方法， 将原始数据的 列 直接转换成 MultiIndex

data_ser = pop_flat.set_index(['state', 'year', 'population'])
data_ser # Series shape(6,)

Empty DataFrame
Columns: []
Index: [(California, 2000, 33871648), (California, 2010, 37253956), (New York, 2000, 18976457), (New York, 2010, 19378102), (Texas, 2000, 20851820), (Texas, 2010, 25145561)]

In [80]:
data_ser.shape
type(data_ser)

pandas.core.frame.DataFrame

In [70]:
# DataFrame shape(6,1), set_index()方法可以设置多级索引

df_x = pop_flat.set_index(['state', 'year'])

df_x

population
state      year            
California 2000    33871648
           2010    37253956
New York   2000    18976457
           2010    19378102
Texas      2000    20851820
           2010    25145561

In practice, I find this type of reindexing to be one of the more useful patterns when encountering real-world datasets.

## 3.6.5 Data Aggregations on Multi-Indices / 多级索引的数据累计方法

We've previously seen that Pandas has built-in data aggregation methods, such as ``mean()``, ``sum()``, and ``max()``.
For hierarchically indexed data, these can be passed a ``level`` parameter that controls which subset of the data the aggregate is computed on.

For example, let's return to our health data:

In [71]:
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      38.0  36.6  21.0  37.9  36.0  37.7
     2      29.0  37.5  55.0  36.4  47.0  37.3
2014 1      23.0  36.2  51.0  36.9  30.0  35.2
     2      44.0  37.4  42.0  37.4  21.0  36.0

Perhaps we'd like to average-out the measurements in the two visits each year. We can do this by naming the index level we'd like to explore, in this case the year:

In [82]:
# 🐍 默认按行累计 (axis=0)

data_mean0 = health_data.mean(axis=0)
data_mean0

subject  type
Bob      HR      30.750
         Temp    37.275
Guido    HR      37.000
         Temp    36.250
Sue      HR      37.000
         Temp    37.375
dtype: float64

By further making use of the ``axis`` keyword, we can take the mean among levels on the columns as well:

In [83]:
# 🐍 axis=1 按列累计

data_mean1 = health_data.mean(axis=1)
data_mean1

year  visit
2013  1        36.366667
      2        37.650000
2014  1        34.183333
      2        35.566667
dtype: float64

Thus in two lines, we've been able to find the average heart rate and temperature measured among all subjects in all visits each year.
This syntax is actually a short cut to the ``GroupBy`` functionality, which we will discuss in [Aggregation and Grouping](03.08-Aggregation-and-Grouping.ipynb).
While this is a toy example, many real-world datasets have similar hierarchical structure.

## Aside: Panel Data / Panel 数据

Pandas has a few other fundamental data structures that we have not yet discussed, namely the ``pd.Panel`` and ``pd.Panel4D`` objects.
These can be thought of, respectively, as three-dimensional and four-dimensional generalizations of the (one-dimensional) ``Series`` and (two-dimensional) ``DataFrame`` structures.
Once you are familiar with indexing and manipulation of data in a ``Series`` and ``DataFrame``, ``Panel`` and ``Panel4D`` are relatively straightforward to use.
In particular, the ``ix``, ``loc``, and ``iloc`` indexers discussed in [Data Indexing and Selection](03.02-Data-Indexing-and-Selection.ipynb) extend readily to these higher-dimensional structures.

We won't cover these panel structures further in this text, as I've found in the majority of cases that multi-indexing is a more useful and conceptually simpler representation for higher-dimensional data.
Additionally, panel data is fundamentally a dense data representation, while multi-indexing is fundamentally a sparse data representation.
As the number of dimensions increases, the dense representation can become very inefficient for the majority of real-world datasets.
For the occasional specialized application, however, these structures can be useful.
If you'd like to read more about the ``Panel`` and ``Panel4D`` structures, see the references listed in [Further Resources](03.13-Further-Resources.ipynb).

<!--NAVIGATION-->
< [Handling Missing Data](03.04-Missing-Values.ipynb) | [Contents](Index.ipynb) | [Combining Datasets: Concat and Append](03.06-Concat-And-Append.ipynb) >